In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviews_Project") \
    .getOrCreate()

In [ ]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, BooleanType

file_path = "/content/drive/MyDrive/Electronics.json.gz"

# read raw lines
df_text = spark.read.text(file_path)

print("Raw lines:", df_text.count())

Raw lines: 20994353


In [ ]:
review_schema = StructType([
    StructField("asin", StringType(), True),
    StructField("overall", DoubleType(), True),
    StructField("reviewText", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("unixReviewTime", LongType(), True),
    StructField("verified", BooleanType(), True)
])

In [ ]:
df_raw = df_text.select(
    from_json(col("value"), review_schema).alias("data")
).select("data.*")

print("Parsed rows:", df_raw.count())

Parsed rows: 20994353


In [ ]:
df_small = df_raw.limit(50000)

print("Limited rows:", df_small.count())

Limited rows: 50000


In [ ]:
from pyspark.sql.functions import col

df_small = df_small.withColumn("label", col("overall").cast("int"))

df_small = df_small.filter(col("reviewText").isNotNull())

print("Rows after cleaning:", df_small.count())

Rows after cleaning: 49989


In [ ]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline

tokenizer = Tokenizer(inputCol="reviewText", outputCol="words")

remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered"
)

hashingTF = HashingTF(
    inputCol="filtered",
    outputCol="rawFeatures",
    numFeatures=2000   # small and safe
)

idf = IDF(
    inputCol="rawFeatures",
    outputCol="features"
)

pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf])

model_small = pipeline.fit(df_small)

df_features = model_small.transform(df_small)

print("Feature transformation complete.")

Feature transformation complete.


In [ ]:
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=5
)

lr_model = lr.fit(train_df)

print("Logistic Regression trained.")

Logistic Regression trained.


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

predictions_lr = lr_model.transform(test_df)

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy_lr = evaluator.evaluate(predictions_lr)

print("Logistic Regression Accuracy:", accuracy_lr)

Logistic Regression Accuracy: 0.6518363690653927


In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=5
)

dt_model = dt.fit(train_df)

predictions_dt = dt_model.transform(test_df)
accuracy_dt = evaluator.evaluate(predictions_dt)

print("Decision Tree Accuracy:", accuracy_dt)

Decision Tree Accuracy: 0.6133174081815467


In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=10,
    maxDepth=5
)

rf_model = rf.fit(train_df)

predictions_rf = rf_model.transform(test_df)
accuracy_rf = evaluator.evaluate(predictions_rf)

print("Random Forest Accuracy:", accuracy_rf)

Random Forest Accuracy: 0.6081417338509008


In [ ]:
from pyspark.ml.classification import NaiveBayes

nb = NaiveBayes(
    featuresCol="features",
    labelCol="label"
)

nb_model = nb.fit(train_df)

predictions_nb = nb_model.transform(test_df)
accuracy_nb = evaluator.evaluate(predictions_nb)

print("Naive Bayes Accuracy:", accuracy_nb)

Naive Bayes Accuracy: 0.11665173683686672


In [ ]:
df_small.groupBy("label").count().orderBy("label").show()

+-----+-----+
|label|count|
+-----+-----+
|    1| 5291|
|    2| 2390|
|    3| 3478|
|    4| 8502|
|    5|30328|
+-----+-----+



In [ ]:
df_small.groupBy("verified").count().show()

+--------+-----+
|verified|count|
+--------+-----+
|    true|39555|
|   false|10434|
+--------+-----+



In [ ]:
from pyspark.sql.functions import year, from_unixtime

df_small = df_small.withColumn("year", year(from_unixtime("unixReviewTime")))

df_small.groupBy("year").count().orderBy("year").show()

+----+-----+
|year|count|
+----+-----+
|1997|    1|
|1999|  636|
|2000| 1620|
|2001|  703|
|2002|  472|
|2003|  340|
|2004|  301|
|2005|  450|
|2006|  696|
|2007| 1135|
|2008| 1090|
|2009| 1170|
|2010| 1805|
|2011| 2416|
|2012| 3229|
|2013| 6325|
|2014| 8806|
|2015| 8894|
|2016| 5696|
|2017| 3498|
+----+-----+
only showing top 20 rows


In [ ]:
pdf = df_small.select("reviewText", "label").limit(10000).toPandas()

print(pdf.shape)
pdf.head()

(10000, 2)


,reviewText,label
0,This was the first time I read Garcia-Aguilera...,5
1,"As with all of Ms. Garcia-Aguilera's books, I ...",5
2,I've not read any of Ms Aguilera's works befor...,5
3,This romance novel is right up there with the ...,4
4,Carolina Garcia Aguilera has done it again. S...,5


In [ ]:
from sklearn.model_selection import train_test_split

X = pdf["reviewText"]
y = pdf["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 8000
Test size: 2000


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

vectorizer = TfidfVectorizer(
    max_features=2000,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

clf = LogisticRegression(
    max_iter=200,
    n_jobs=-1
)

clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)

accuracy_sklearn = accuracy_score(y_test, y_pred)

print("Scikit-Learn Logistic Regression Accuracy:", accuracy_sklearn)

Scikit-Learn Logistic Regression Accuracy: 0.67


In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.classification import LogisticRegression

# Base Logistic Regression model
lr_cv = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

# Hyperparameter grid (small and safe)
paramGrid = (ParamGridBuilder()
             .addGrid(lr_cv.regParam, [0.0, 0.1])
             .addGrid(lr_cv.elasticNetParam, [0.0, 0.5])
             .build())

# Evaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

# CrossValidator with parallelism
crossval = CrossValidator(
    estimator=lr_cv,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2
)

# Fit CV model
cv_model = crossval.fit(train_df)

# Evaluate on test set
predictions_cv = cv_model.transform(test_df)

accuracy_cv = evaluator.evaluate(predictions_cv)

print("Cross-Validated Logistic Regression Accuracy:", accuracy_cv)

Cross-Validated Logistic Regression Accuracy: 0.6354135562854584


In [ ]:
df_export = df_small.select(
    "label",
    "reviewText",
    "verified",
    "year"
)

df_export.limit(20000).toPandas().to_csv("amazon_tableau_sample.csv", index=False)

In [ ]:
!ls

amazon_tableau_sample.csv  drive  sample_data


In [ ]:
from google.colab import files
files.download("amazon_tableau_sample.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>